<a href="https://colab.research.google.com/github/hureadx/TypeRip/blob/master/2026_03_27_%E5%AE%8C%E6%88%90%E5%93%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================
# Colab: タイムラインCSV(1hオフセット対応) → インメモリ音声処理 → Whisper(モデル指定可)
#     → 機械ルールで禁則/幅/改行/記号処理/空対応/☆付与 → SRT
# =========================================

# 必要なライブラリのインストール（Colab想定）
!pip -q install -U openai-whisper pydub

import csv
import datetime
import json
import os
import re
import unicodedata
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import whisper
from pydub import AudioSegment

NL = chr(10)

# Colab判定
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False


# -----------------------------
# 1) パスの設定
# -----------------------------

MP3_PATH = "/content/drive/MyDrive/video/オーディオ.mp3"
CSV_PATH = "/content/drive/MyDrive/video/Timeline-1-copy-1.csv"

# ルール設定ファイル（任意）
RULES_JSON_PATH = "/content/drive/MyDrive/video/subtitle_rules.json"

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)


# -----------------------------
# 2) 設定データ型（★変更可能パラメータをここに集約）
# -----------------------------

@dataclass
class ProcessingConfig:
    # 音声・タイムライン処理
    fps: float = 60.0                    # ★タイムコード変換のフレームレート（30.0 / 60.0 / 29.97 など）
    whisper_model_name: str = "large-v3"    # ★Whisperモデル名: "tiny", "base", "small", "medium", "large-v3", "turbo"
    whisper_language: str = "ja"         # 認識言語
    whisper_compression_threshold: float = 1.0  # ★Whisperセグメント分割の閾値（小さいほど細かく分割、デフォルト2.4）

    # タイムラインオフセット（1時間=3600秒など）
    timeline_offset_seconds: float = 3600.0


@dataclass
class ArtifactSaveConfig:
    save_whisper_raw_json: bool = False   # whisperJson保存切り替え


# 設定インスタンス（ここで値を変更してください）
CONFIG = ProcessingConfig()
SAVE_ARTIFACTS = ArtifactSaveConfig()


@dataclass
class RawSegment:
    id: int
    clip_start_offset: float
    start: float
    end: float
    text: str


@dataclass
class SubtitleRules:
    # 表示
    max_lines: int = 2
    max_width: int = 32  # 全角=2, 半角=1 の「表示幅」
    min_second_line_width: int = 4

    # 記号
    banned_chars: str = "？！「」（）()[]{}"
    remove_chars: str = ""

    # 出力
    empty_replacement: str = ";"
    retry_symbol: str = "☆"

    # 改行戦略
    prefer_sentence_marker: bool = True
    prefer_break_marker: bool = True


# -----------------------------
# 3) ユーティリティ
# -----------------------------

def timecode_to_seconds(tc: str, fps: float) -> float:
    """タイムコード文字列を秒数に変換（fps指定版）"""
    if not tc or not isinstance(tc, str):
        return 0.0
    parts = tc.split(":")
    if len(parts) != 4:
        return 0.0
    try:
        h, m, s, f = map(int, parts)
    except Exception:
        return 0.0
    return h * 3600 + m * 60 + s + f / fps


def normalize_ws(s: str) -> str:
    if not s:
        return ""
    s = s.replace("　", " ")
    return " ".join(s.split()).strip()


def display_width(text: str) -> int:
    """字幕の見た目幅: 全角寄り(F/W/A)=2, それ以外=1"""
    w = 0
    for ch in text:
        if ch == NL:
            continue
        eaw = unicodedata.east_asian_width(ch)
        if eaw in ("F", "W", "A"):
            w += 2
        else:
            w += 1
    return w


def strip_banned(text: str, rules: SubtitleRules) -> str:
    if not text:
        return ""
    table = str.maketrans({c: "" for c in (rules.banned_chars + rules.remove_chars)})
    return text.translate(table)


def safe_join(line: str) -> str:
    return normalize_ws(line)


def build_plain_to_full_map(full: str) -> List[int]:
    """full(マーカーなし) の各文字位置 -> full index を返す"""
    return list(range(len(full)))


def choose_break_index(full: str, rules: SubtitleRules) -> Optional[int]:
    """2行に割るための break index を返す"""
    plain = full
    total_w = display_width(plain)
    if total_w <= rules.max_width:
        return None

    target_w = min(rules.max_width, max(rules.max_width - 2, total_w // 2))

    candidates: List[int] = []

    for i, ch in enumerate(plain):
        if ch == " ":
            candidates.append(i + 1)

    particles = [
        "は", "が", "を", "に", "で", "と", "も", "や", "へ",
        "から", "まで", "より", "って", "けど", "でも", "だから", "ほんで", "そやけど",
    ]
    for p in particles:
        start = 0
        while True:
            idx = plain.find(p, start)
            if idx == -1:
                break
            cut = idx + len(p)
            if 0 < cut <= len(plain):
                candidates.append(cut)
            start = idx + 1

    uniq: List[int] = []
    seen = set()
    for c in candidates:
        c = max(0, min(len(plain), c))
        if c in seen:
            continue
        seen.add(c)
        uniq.append(c)

    def width_left(c: int) -> int:
        return display_width(safe_join(plain[:c]))

    def width_right(c: int) -> int:
        return display_width(safe_join(plain[c:]))

    feasible: List[Tuple[int, int, int]] = []
    for c in uniq:
        wl = width_left(c)
        wr = width_right(c)
        if wl <= rules.max_width and wr <= rules.max_width:
            feasible.append((c, wl, wr))

    if feasible:
        def score(item: Tuple[int, int, int]) -> float:
            c, wl, wr = item
            s = -abs(wl - target_w)
            if wr < rules.min_second_line_width:
                s -= 50
            if wl == rules.max_width:
                s -= 1
            return s

        best = max(feasible, key=score)
        return best[0]

    return None


def apply_rules_to_text(whisper_text: str, retry: bool, rules: SubtitleRules) -> str:
    text = normalize_ws(whisper_text or "")

    text = strip_banned(text, rules)

    if not text:
        out = rules.empty_replacement
    else:
        idx = choose_break_index(text, rules)
        if idx is None:
            out = safe_join(text)
        else:
            left = safe_join(text[:idx])
            right = safe_join(text[idx:])
            if left and right:
                out = left + NL + right
            else:
                out = left or right or rules.empty_replacement

    lines = [ln for ln in out.split(NL) if ln.strip()]
    if not lines:
        lines = [rules.empty_replacement]
    if len(lines) > rules.max_lines:
        lines = lines[: rules.max_lines]
    out = NL.join(lines)

    if retry and not out.startswith(rules.retry_symbol):
        out = rules.retry_symbol + out

    return out.strip()


# -----------------------------
# 4) 音声処理
# -----------------------------

def parse_timeline_csv(csv_path: str, offset_seconds: float, fps: float) -> List[Tuple[float, float]]:
    """タイムラインCSVを解析してクリップ区間を抽出（fps指定版）"""
    clips: List[Tuple[float, float]] = []
    seen_intervals = set()

    if not os.path.exists(csv_path):
        print("CSVファイルが見つかりません:", csv_path)
        return []

    with open(csv_path, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            track = row.get("A", "")
            if isinstance(track, str) and track.startswith("A"):
                continue

            start_tc = row.get("タイムライン上のイン")
            end_tc = row.get("タイムライン上のアウト")
            if start_tc and end_tc:
                start_s = timecode_to_seconds(start_tc, fps) - offset_seconds
                end_s = timecode_to_seconds(end_tc, fps) - offset_seconds
                start_s = max(0.0, start_s)
                end_s = max(0.0, end_s)
                interval = (round(start_s, 3), round(end_s, 3))
                if interval[1] > interval[0] and interval not in seen_intervals:
                    clips.append((start_s, end_s))
                    seen_intervals.add(interval)

    clips.sort()
    return clips


def transcribe_clips(
    audio_path: str,
    clips: List[Tuple[float, float]],
    model_name: str,
    language: str,
    compression_threshold: float,
) -> Tuple[List[RawSegment], List[Dict[str, Any]]]:
    print(f"Whisperモデルを読み込んでいます: {model_name}...")
    model = whisper.load_model(model_name)

    print("音声ファイルを読み込み中...")
    audio = AudioSegment.from_file(audio_path)
    audio = audio.set_frame_rate(16000).set_channels(1)
    audio_duration_ms = len(audio)

    all_raw: List[RawSegment] = []
    whisper_full: List[Dict[str, Any]] = []
    gid = 0

    for i, (start_s, end_s) in enumerate(clips):
        start_ms = int(start_s * 1000)
        end_ms = int(end_s * 1000)

        if start_ms >= audio_duration_ms:
            continue

        print("  クリップ", i + 1, "/", len(clips), "を処理中 (", round(start_s, 1), "s -", round(end_s, 1), "s )")

        actual_end_ms = min(end_ms, audio_duration_ms)
        clip_audio = audio[start_ms:actual_end_ms]

        if len(clip_audio) < 100:
            continue

        samples = np.array(clip_audio.get_array_of_samples()).astype(np.float32) / 32768.0

        result = model.transcribe(
            samples,
            language=language,
            compression_ratio_threshold=compression_threshold,
        )

        whisper_full.append({
            "clip_index": i,
            "timeline_start": start_s,
            "timeline_end": end_s,
            "whisper_output": result,
        })

        for s in result.get("segments", []):
            text = normalize_ws(str(s.get("text") or ""))
            if not text:
                continue
            all_raw.append(RawSegment(
                id=gid,
                clip_start_offset=start_s,
                start=float(s.get("start", 0.0)),
                end=float(s.get("end", 0.0)),
                text=text,
            ))
            gid += 1

    return all_raw, whisper_full


# -----------------------------
# 5) ファイル出力
# -----------------------------

def seconds_to_srt_timestamp(t: float) -> str:
    total_ms = int(round(t * 1000))
    ms = total_ms % 1000
    s = (total_ms // 1000) % 60
    m = (total_ms // 60000) % 60
    h = total_ms // 3600000
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def format_srt_text(srt_text: str) -> str:
    lines = [ln.strip() for ln in srt_text.splitlines() if ln.strip()]

    out: List[str] = []
    cue_count = 0
    i = 0
    while i < len(lines):
        line = lines[i]
        is_id = line.isdigit()
        is_time = (i + 1 < len(lines)) and ("-->" in lines[i + 1])
        if is_id and is_time:
            cue_count += 1
            if cue_count >= 2:
                out.append("")
            out.append(line)
            i += 1
            continue
        out.append(line)
        i += 1

    return NL.join(out) + NL


def export_files(segs: List[RawSegment], base_path: str, offset_to_restore: float):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    srt_path = base_path + "_" + timestamp + ".srt"

    buf: List[str] = []
    for i, s in enumerate(segs, 1):
        final_start = s.clip_start_offset + s.start + offset_to_restore
        final_end = s.clip_start_offset + s.end + offset_to_restore
        buf.append(str(i))
        buf.append(seconds_to_srt_timestamp(final_start) + " --> " + seconds_to_srt_timestamp(final_end))
        buf.append(s.text)
        buf.append("")

    srt_text = NL.join(buf)
    srt_text = format_srt_text(srt_text)

    with open(srt_path, "w", encoding="utf-8") as f:
        f.write(srt_text)

    print(NL + "字幕ファイルを書き出しました: " + srt_path)


# -----------------------------
# 6) メイン
# -----------------------------

def main():
    # 設定の確認表示
    print("[Config] fps =", CONFIG.fps)
    print("[Config] whisper_model =", CONFIG.whisper_model_name)
    print("[Config] whisper_language =", CONFIG.whisper_language)
    print("[Config] whisper_compression_threshold =", CONFIG.whisper_compression_threshold)
    print("[Config] timeline_offset =", CONFIG.timeline_offset_seconds, "sec")

    rules = load_rules(RULES_JSON_PATH)
    print("[Rules]", rules)

    base_dir = os.path.dirname(MP3_PATH)
    now_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    output_base = os.path.join(base_dir, "subtitle_" + now_str)

    print(NL + "[1/4] タイムラインCSVを解析しています...")
    clips = parse_timeline_csv(
        CSV_PATH,
        offset_seconds=CONFIG.timeline_offset_seconds,
        fps=CONFIG.fps
    )
    print("検出されたクリップ数:", len(clips))

    if not clips:
        return

    print(NL + "[2/4] Whisperで音声を文字起こししています...")
    raw_segments, whisper_raw = transcribe_clips(
        MP3_PATH,
        clips,
        model_name=CONFIG.whisper_model_name,
        language=CONFIG.whisper_language,
        compression_threshold=CONFIG.whisper_compression_threshold,
    )

    whisper_json_path = output_base + "_whisper_raw.json"
    if SAVE_ARTIFACTS.save_whisper_raw_json:
        with open(whisper_json_path, "w", encoding="utf-8") as f:
            json.dump(whisper_raw, f, ensure_ascii=False, indent=2)
        print("Whisperの生データを保存しました:", whisper_json_path)

    if not raw_segments:
        print(NL + "[エラー] 有効な音声セグメントが見つかりませんでした")
        return

    print(NL + "[3/4] 機械ルールで字幕を整形しています...")
    for s in raw_segments:
        s.text = apply_rules_to_text(s.text, retry=False, rules=rules)

    print(NL + "[4/4] タイムスタンプを復元して書き出します...")
    export_files(raw_segments, output_base, offset_to_restore=CONFIG.timeline_offset_seconds)


def load_rules(path: str) -> SubtitleRules:
    if path and os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            base = asdict(SubtitleRules())
            for k, v in data.items():
                if k in base:
                    base[k] = v
            return SubtitleRules(**base)
        except Exception as e:
            print("[WARN] rules.json の読み込みに失敗。デフォルトで続行:", e)
    return SubtitleRules()


if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 89.9 MB/s eta 0:00:00


OSError: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchaudio/lib/_torchaudio.abi3.so